In [0]:
# Databricks notebook source
# MAGIC 

In [0]:
%run ./shared_audit_utils

In [0]:
# Import libraries
from pyspark.sql.functions import *
from pyspark.sql import DataFrame
import logging
from datetime import datetime

2026-06-04 22:14:27,237 INFO Received command c on object id p0


2026-06-04 22:14:27,343 INFO Received command c on object id p0


DataFrame[]

2026-06-04 22:14:28,247 INFO Received command c on object id p0


In [0]:
# Run id
run_id = get_run_id()
start_time = datetime.now()

+------------------------------------+--------------------+--------------------------+--------------------------+-------+-----------+------------+-------------+
|run_id                              |pipeline_name       |start_time                |end_time                  |status |input_count|output_count|error_message|
+------------------------------------+--------------------+--------------------------+--------------------------+-------+-----------+------------+-------------+
|755225e1-58c8-4bb5-8fdf-b0a9da8222f6|bronze_online_retail|2026-06-04 22:13:54.464814|2026-06-04 22:14:04.398468|SUCCESS|541909     |541909      |NULL         |
+------------------------------------+--------------------+--------------------------+--------------------------+-------+-----------+------------+-------------+



run_id,pipeline_name,start_time,end_time,status,input_count,output_count,error_message
755225e1-58c8-4bb5-8fdf-b0a9da8222f6,bronze_online_retail,2026-06-04T22:13:54.464Z,2026-06-04T22:14:04.398Z,SUCCESS,541909,541909,null


In [0]:
# Logging cofiguration
logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s %(levelname)s %(message)s"
)
logger = logging.getLogger("silver_pipeline")

In [0]:
# Configuration
PIPELINE_NAME = "silver_online_pipeline"
BRONZE_TABLE = "workspace.default.bronze_online_retail"
SILVER_TABLE = "workspace.default.silver_online_retail"

In [0]:
# Read source
def read_bronze() -> DataFrame:
    logger.info(f"Reading bronze table: {BRONZE_TABLE}")
    return (
        spark.table(BRONZE_TABLE)
    )
                

In [0]:
# Validation
def validate_source(df:DataFrame):
    logger.info("Running source validation")
    if df.limit(1).count() == 0:
        raise Exception(
            f"Source table: {BRONZE_TABLE} is empty"
            )


In [0]:
# Remove cancelled orders
def remove_cancelled_orders(df:DataFrame) -> DataFrame:
    logger.info("Removing cancelled orders")
    return (
        df.filter(~col("InvoiceNo").startswith("C"))
    )

In [0]:
# Remove Invalide Quqntity
def remove_invalid_quantity(df:DataFrame) -> DataFrame:
    logger.info("Removing invalid quantity")
    return (
        df.filter(col("Quantity") > 0)
    )

In [0]:
# Standardize Date
def convert_invoice_date(df:DataFrame) -> DataFrame:
    logger.info("Converting InvoiceDate")
    return (
        df.withColumn("InvoiceDate", try_to_timestamp(col("InvoiceDate"), lit("dd-MM-yyyy HH:mm")))
    )

In [0]:
# Validate dates
def validate_dates(df: DataFrame):
    invalid_dates = (
        df.filter(col("InvoiceDate").isNull()).count()
    )
    logger.info(f"Invalid dates found: {invalid_dates}")
    if invalid_dates > 0:
        raise Exception(
            f"{invalid_dates} invalid dates detected"
        )

2026-06-04 22:14:35,797 INFO Received command c on object id p0


In [0]:
# Business column
def add_sales_amount(df:DataFrame) -> DataFrame:
    logger.info("Adding sales amount")
    return (
        df.withColumn("SalesAmount", round(col("Quantity") * col("UnitPrice"), 2))
    )



2026-06-04 22:14:36,019 INFO Received command c on object id p0


In [0]:
# Metadata columns
def add_metadata(df: DataFrame) -> DataFrame:

    return (
        df
        .withColumn("load_timestamp",current_timestamp())
        .withColumn("pipeline_name",lit(PIPELINE_NAME))
    )

2026-06-04 22:14:36,204 INFO Received command c on object id p0


In [0]:
# Write to silver
def write_silver(df:DataFrame):
    logger.info(f"Writing silver table: {SILVER_TABLE}")
    return (
        df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
    )


In [0]:
# Audit Metrics
def log_pipeline_metrics(input_count,output_count,df):
    logger.info(f"Input Count: {input_count}")
    logger.info(f"Output Count: {output_count}")
    logger.info(f"Rejected Count: {input_count-output_count}")
    logger.info(
        f"Null CustomerID: "
        f"{df.filter(col('CustomerID').isNull()).count()}"
    )
    logger.info(
        f"Null Description: "
        f"{df.filter(col('Description').isNull()).count()}"
    )

In [0]:
# Main
def main(run_id,start_time):
    logger.info("Starting silver pipeline")
    bronze_df = read_bronze()
    validate_source(bronze_df)
    input_count = bronze_df.count()
    silver_df = (bronze_df
                 .transform(remove_cancelled_orders)
                 .transform(remove_invalid_quantity)
                 .transform(convert_invoice_date)
                 .transform(add_sales_amount)
                 .transform(add_metadata))
    
    output_count = silver_df.count()
    validate_dates(silver_df)
    log_pipeline_metrics(input_count,output_count,silver_df)
    write_silver(silver_df)
    logger.info("Finished silver pipeline")

    end_time = datetime.now()
    # Write audit record
    # logger.info("Writing logs to audit table")
    write_audit_record(
        run_id=run_id,
        pipeline_name=PIPELINE_NAME,
        start_time=start_time,
        end_time=end_time,
        status="SUCCESS",
        input_count=input_count,
        output_count=output_count,
        error_message=None
    )

In [0]:
# Entry Point
if __name__ == "__main__":

    run_id = get_run_id()
    start_time = datetime.now()

    try:

        main(
            run_id,
            start_time
        )

    except Exception as e:

        end_time = datetime.now()

        write_audit_record(
            run_id=run_id,
            pipeline_name=PIPELINE_NAME,
            start_time=start_time,
            end_time=end_time,
            status="FAILED",
            input_count=0,
            output_count=0,
            error_message=str(e)
        )

        logger.exception(
            f"Pipeline {PIPELINE_NAME} failed: {e}"
        )

        raise

2026-06-04 22:14:47,112 INFO Starting silver pipeline
2026-06-04 22:14:47,119 INFO Reading bronze table: workspace.default.bronze_online_retail
2026-06-04 22:14:47,120 INFO Running source validation
2026-06-04 22:14:48,535 INFO Removing cancelled orders
2026-06-04 22:14:48,536 INFO Removing invalid quantity
2026-06-04 22:14:48,537 INFO Converting InvoiceDate
2026-06-04 22:14:48,538 INFO Adding sales amount
2026-06-04 22:14:50,319 INFO Invalid dates found: 0
2026-06-04 22:14:50,320 INFO Input Count: 541909
2026-06-04 22:14:50,320 INFO Output Count: 531285
2026-06-04 22:14:50,320 INFO Rejected Count: 10624
2026-06-04 22:14:50,997 INFO Null CustomerID: 133361
2026-06-04 22:14:51,601 INFO Null Description: 592
2026-06-04 22:14:51,602 INFO Writing silver table: workspace.default.silver_online_retail
2026-06-04 22:14:54,427 INFO Finished silver pipeline
2026-06-04 22:14:54,462 INFO Writing logs to audit table
